# NLP 课堂实践：构建React Agent系统

假设你正在准备一次旅行，想知道“下周去杭州适不适合露营”。如果只是问搜索引擎，它可能会返回一系列网页，需要我们自己筛选、判断和整合信息；如果只是调用大模型，它通常会基于自身已经学习到的参数化知识来生成一个看起来很完整的建议。

但这里有一个问题：大模型的参数化知识主要来自训练阶段，它并不天然等于实时信息。比如下周杭州的天气、某个露营地是否临时关闭、交通是否管制、当地政策有没有变化，这些信息都可能是动态变化的。只依赖模型“记住的知识”，就很难保证决策足够可靠。

真正可靠的决策，往往不是“一次回答”就完成的。

一个更像人类的思考过程应该是这样的：
先判断这个问题需要哪些信息，比如天气、地点、交通、露营政策；然后去查询实时天气；发现还需要确认营地是否开放；再继续搜索；最后把查到的信息综合起来，给出建议。

也就是说，我们并不是简单地“问一次、答一次”，而是在不断经历：

思考 → 行动 → 观察 → 再思考 → 再行动

这正是 ReAct Agent 的核心思想。

## 实践目录
1. **React基础知识**：掌握React Agent基本思想和设计结构
2. **React Agent环境配置与初步代码实现**：掌握React Agent环境部署、配置大模型API、动手实践基本的react联网查询应用。
3. **讨论**：通过实践学习，总结思考如何进一步优化agent

## React Agent 基础知识
Agent作为能够感知环境、做出决策并采取行动以实现特定目标的自主实体。与传统直接调用语言模型相比，Agent 具备以下核心特征：
* 自主性：无需人工干预即可独立运行
* 反应性：能对环境变化做出实时响应
* 主动性：主动追求目标而非被动响应
* 社会性：像人类进行交互

React Agent = Reasoning + Acting， 即不让LLM一次性回答出问题的最终答案，而是让它不断地与外界真实环境和工具进行交互，通过多次“思考”和“行动”，根据环境反馈，再进行下一步决策
 

ReAct 的机制可以理解为让 LLM 在任务执行中不断循环“思考—行动—观察”：模型先根据当前问题进行 Thought，判断下一步该做什么；如果需要外部信息，就通过 Function Call 调用工具，或直接向环境发出 Action；随后环境或工具返回 Observation，模型再根据新的观察结果继续推理和决策，直到得到最终答案。它的核心不是一次性生成结果，而是让模型在推理和行动之间迭代完成任务。

React每一轮迭代LLM输出结构：

(1) Thought: 思考为什么要这么做（LLM 内部推理）

(2) Action: 调用哪些 Tool（函数名，例如 "search"）

(3) Observation: 工具返回运行结果


![图片说明](image.png)



In [1]:
LLM_API_KEY="sk-poicqmpadgbiuskfgobkbzajbpvgxfactefvfkgvjowpbzvm"
LLM_MODEL_ID="glm-4.7-flash"
LLM_BASE_URL="https://open.bigmodel.cn/api/paas/v4"
SERPAPI_API_KEY = "bf65d14dadbc63ec249173135d72c3e5b325386a73a305da5dc20655e3bdadc1"

In [2]:
!pip install openai
!pip install google-search-results

In [3]:
import os
from openai import OpenAI
from dotenv import load_dotenv
from typing import List, Dict

# 加载 .env 文件中的环境变量
load_dotenv()

class HelloAgentsLLM:
    """
    设计请求LLM
    """
    def __init__(self, model: str = None, apiKey: str = None, baseUrl: str = None, timeout: int = None):
        """
        初始化 大语言模型请求 客户端，优先使用传入参数，否则从环境变量读取模型、API Key、服务地址和超时时间。
        """
        self.model = model or os.getenv("LLM_MODEL_ID")
        apiKey = apiKey or os.getenv("LLM_API_KEY")
        baseUrl = baseUrl or os.getenv("LLM_BASE_URL")
        timeout = timeout or int(os.getenv("LLM_TIMEOUT", 60))
        
        if not all([self.model, apiKey, baseUrl]):
            raise ValueError("模型ID、API密钥和服务地址必须被提供或在.env文件中定义。")

        self.client = OpenAI(api_key=apiKey, base_url=baseUrl, timeout=timeout)

    def think(self, messages: List[Dict[str, str]], temperature: float = 0) -> str:
        """
        调用大语言模型接口，流式输出模型回复，并将完整回复内容拼接后返回。
        """
        print(f" 正在调用 {self.model} 模型...")
        try:
            response = self.client.chat.completions.create(
                model=self.model,
                messages=messages,
                temperature=temperature,
                stream=True,
            )
            
            # 处理流式响应
            print(" 大语言模型响应成功:")
            collected_content = []
            for chunk in response:
                if not chunk.choices:
                    continue
                content = chunk.choices[0].delta.content or ""
                print(content, end="", flush=True)
                collected_content.append(content)
            print()  # 在流式输出结束后换行
            return "".join(collected_content)

        except Exception as e:
            print(f" 调用LLM API时发生错误: {e}")
            return None


In [4]:
if __name__ == "__main__":
    llm = HelloAgentsLLM()
    reply = llm.think([
        {"role": "user", "content": "请用一句话自我介绍。"},
    ])
    print("\n--- 测试结果 ---")
    print("通畅" if reply else "失败")


 正在调用 zai-org/GLM-4.5-Air 模型...
 大语言模型响应成功:
我是个以好奇为眼、以真诚为心，在平凡日常里探索美好、传递温度的人。

--- 测试结果 ---
通畅


In [5]:
# ReAct 提示词模板
REACT_PROMPT_TEMPLATE = """
请注意，你是一个有能力调用外部工具的智能助手。

可用工具如下:
{tools}

请严格按照以下格式进行回应:

Thought: 你的思考过程，用于分析问题、拆解任务和规划下一步行动。
Action: 你决定采取的行动，必须是以下格式之一:
- `{{tool_name}}[{{tool_input}}]`:调用一个可用工具。
- `Finish[最终答案]`:当你认为已经获得最终答案时。
- 当你收集到足够的信息，能够回答用户的最终问题时，你必须在Action:字段后使用 Finish[最终答案] 来输出最终答案。

现在，请开始解决以下问题:
Question: {question}
History: {history}
"""

In [6]:
class ToolExecutor:
    """
    一个工具执行器，负责管理和执行工具。
    """
    def __init__(self):
        self.tools: Dict[str, Dict[str, Any]] = {}

    def registerTool(self, name: str, description: str, func: callable):
        """
        向工具箱中注册一个新工具。
        """
        if name in self.tools:
            print(f"警告：工具 '{name}' 已存在，将被覆盖。")
        
        self.tools[name] = {"description": description, "func": func}
        print(f"工具 '{name}' 已注册。")

    def getTool(self, name: str) -> callable:
        """
        根据名称获取一个工具的执行函数。
        """
        return self.tools.get(name, {}).get("func")

    def getAvailableTools(self) -> str:
        """
        获取所有可用工具的格式化描述字符串。
        """
        return "\n".join([
            f"- {name}: {info['description']}" 
            for name, info in self.tools.items()
        ])

In [7]:
from serpapi import SerpApiClient
def search(query: str) -> str:
    """
    联网获取搜索结果
    """
    print(f" 正在执行 [SerpApi] 网页搜索: {query}")
    try:
        api_key = os.getenv("SERPAPI_API_KEY")
        if not api_key:
            return "错误：SERPAPI_API_KEY 未在 .env 文件中配置。"

        params = {
            "engine": "google",
            "q": query,
            "api_key": api_key,
            "gl": "cn",  # 国家代码
            "hl": "zh-cn", # 语言代码
        }
        
        client = SerpApiClient(params)
        results = client.get_dict()
        
        # 智能解析：优先寻找最直接的答案
        if "answer_box_list" in results:
            return "\n".join(results["answer_box_list"])
        if "answer_box" in results and "answer" in results["answer_box"]:
            return results["answer_box"]["answer"]
        if "knowledge_graph" in results and "description" in results["knowledge_graph"]:
            return results["knowledge_graph"]["description"]
        if "organic_results" in results and results["organic_results"]:
            # 如果没有直接答案，则返回前三个有机结果的摘要
            snippets = [
                f"[{i+1}] {res.get('title', '')}\n{res.get('snippet', '')}"
                for i, res in enumerate(results["organic_results"][:3])
            ]
            return "\n\n".join(snippets)
        
        return f"对不起，没有找到关于 '{query}' 的信息。"

    except Exception as e:
        return f"搜索时发生错误: {e}"

In [8]:
# --- 工具初始化与使用示例 ---
if __name__ == '__main__':
    # 1. 初始化工具执行器
    toolExecutor = ToolExecutor()

    # 2. 注册我们的实战搜索工具
    search_description = "一个网页搜索引擎。当你需要回答关于时事、事实以及在你的知识库中找不到的信息时，应使用此工具。"
    toolExecutor.registerTool("Search", search_description, search)
    
    # 3. 打印可用的工具
    print("\n--- 可用的工具 ---")
    print(toolExecutor.getAvailableTools())

    # 4. 智能体的Action调用，这次我们问一个实时性的问题
    print("\n--- 执行 Action: Search['英伟达最新的GPU型号是什么'] ---")
    tool_name = "Search"
    tool_input = "英伟达最新的GPU型号是什么"

    tool_function = toolExecutor.getTool(tool_name)
    if tool_function:
        observation = tool_function(tool_input)
        print("--- 观察 (Observation) ---")
        print(observation)
    else:
        print(f"错误：未找到名为 '{tool_name}' 的工具。")

工具 'Search' 已注册。

--- 可用的工具 ---
- Search: 一个网页搜索引擎。当你需要回答关于时事、事实以及在你的知识库中找不到的信息时，应使用此工具。

--- 执行 Action: Search['英伟达最新的GPU型号是什么'] ---
 正在执行 [SerpApi] 网页搜索: 英伟达最新的GPU型号是什么
--- 观察 (Observation) ---
[1] 比较GeForce 系列最新一代显卡和前代显卡| NVIDIA
比较最新一代RTX 30 系列显卡和前代的RTX 20 系列、GTX 10 和900 系列显卡。查看规格、功能、技术支持等内容。

[2] GeForce RTX 50 系列显卡| NVIDIA
GeForce RTX™ 50 系列GPU 搭载NVIDIA Blackwell 架构，为游戏玩家和创作者带来全新玩法。RTX 50 系列具备强大的AI 算力，带来升级体验和更逼真的画面。

[3] 一文彻底读懂：英伟达GPU分类、架构演进和参数解析
Quadro系列是英伟达专业级GPU产品线，针对商业和专业应用领域进行了优化。常见的产品型号如NVIDIA RTX A6000、A5000等。 Quadro GPU具备强大的计算能力、大 ...


In [9]:
class ReActAgent:
    def __init__(self, llm_client: HelloAgentsLLM, tool_executor: ToolExecutor, max_steps: int = 5):
        self.llm_client = llm_client
        self.tool_executor = tool_executor
        self.max_steps = max_steps
        self.history = []

    def run(self, question: str):
        """
        运行ReAct智能体来回答一个问题。
        """
        self.history = [] # 每次运行时重置历史记录
        current_step = 0

        while current_step < self.max_steps:
            current_step += 1
            print(f"--- 第 {current_step} 步 ---")

            # 1. 格式化提示词
            tools_desc = self.tool_executor.getAvailableTools()
            history_str = "\n".join(self.history)
            prompt = REACT_PROMPT_TEMPLATE.format(
                tools=tools_desc,
                question=question,
                history=history_str
            )

            # 2. 调用LLM进行思考
            messages = [{"role": "user", "content": prompt}]
            response_text = self.llm_client.think(messages=messages)
            
            if not response_text:
                print("错误:LLM未能返回有效响应。")
                break
            # (这段逻辑在 run 方法的 while 循环内)
            # 3. 解析LLM的输出
            thought, action = self._parse_output(response_text)
            
            if thought:
                print(f"思考: {thought}")

            if not action:
                print("警告:未能解析出有效的Action，流程终止。")
                break

            # 4. 执行Action
            if action.startswith("Finish"):
                # 如果是Finish指令，提取最终答案并结束
                m = re.match(r"Finish\[(.*)\]", action, re.DOTALL)
                final_answer = m.group(1).strip() if m else action[len("Finish"):].strip(" :[]\n")
                print(f" 最终答案: {final_answer}")
                return final_answer
            
            tool_name, tool_input = self._parse_action(action)
            if not tool_name or not tool_input:
                # ... 处理无效Action格式 ...
                continue

            print(f" 行动: {tool_name}[{tool_input}]")
            
            tool_function = self.tool_executor.getTool(tool_name)
            if not tool_function:
                observation = f"错误:未找到名为 '{tool_name}' 的工具。"
            else:
                observation = tool_function(tool_input) # 调用真实工具
            print(f" 观察: {observation}")
            
            # 将本轮的Action和Observation添加到历史记录中
            self.history.append(f"Action: {action}")
            self.history.append(f"Observation: {observation}")

        # 循环结束:强制让LLM根据已有历史给出最终答案
        print(f"已达到最大步数({self.max_steps})，强制输出最终答案。")
        history_str = "\n".join(self.history)
        force_prompt = (
            f"你已达到最大思考步数,必须立刻给出最终答案,不能再调用任何工具。\n"
            f"请基于以下历史信息直接回答用户的问题。\n\n"
            f"Question: {question}\n"
            f"History:\n{history_str}\n\n"
            f"请只输出最终答案文本,不要再输出 Thought / Action / Finish 等标记。"
        )
        final_answer = self.llm_client.think(
            messages=[{"role": "user", "content": force_prompt}]
        )
        if final_answer:
            final_answer = final_answer.strip()
            print(f" 最终答案: {final_answer}")
        else:
            print(" 警告:LLM未能给出最终答案。")
        return final_answer

    def _parse_output(self, text: str):
        """解析LLM的输出，提取Thought和Action。
        """
        # Thought: 匹配到 Action: 或文本末尾
        thought_match = re.search(r"Thought:\s*(.*?)(?=\nAction:|$)", text, re.DOTALL)
        # Action: 匹配到文本末尾
        action_match = re.search(r"Action:\s*(.*?)$", text, re.DOTALL)
        thought = thought_match.group(1).strip() if thought_match else None
        action = action_match.group(1).strip() if action_match else None
        return thought, action

    def _parse_action(self, action_text: str):
        """解析Action字符串，提取工具名称和输入。
        """
        match = re.match(r"(\w+)\[(.*)\]", action_text, re.DOTALL)
        if match:
            return match.group(1), match.group(2)
        return None, None


In [10]:
import re
llm = HelloAgentsLLM()
tool_executor = ToolExecutor()
search_desc = "一个网页搜索引擎。当你需要回答关于时事、事实以及在你的知识库中找不到的信息时，应使用此工具。"
tool_executor.registerTool("Search", search_desc, search)
agent = ReActAgent(llm_client=llm, tool_executor=tool_executor)
question = "华为在2026年发布了哪些手机？他们的主要卖点是什么？"
agent.run(question)

工具 'Search' 已注册。
--- 第 1 步 ---
 正在调用 zai-org/GLM-4.5-Air 模型...
 大语言模型响应成功:
Thought: 用户询问华为在2026年发布的手机及其主要卖点。首先需要明确当前时间，因为2026年还未到。我需要使用搜索引擎来查找华为2026年的手机发布计划或相关信息，因为这个问题涉及到未来的产品信息，不在我的知识库范围内。

Action: Search[华为2026年手机发布计划]
思考: 用户询问华为在2026年发布的手机及其主要卖点。首先需要明确当前时间，因为2026年还未到。我需要使用搜索引擎来查找华为2026年的手机发布计划或相关信息，因为这个问题涉及到未来的产品信息，不在我的知识库范围内。
 行动: Search[华为2026年手机发布计划]
 正在执行 [SerpApi] 网页搜索: 华为2026年手机发布计划
 观察: [1] 华为2026年手机发布规划曝光：华为三折叠或9月发布
9月，华为三折叠手机将发布。其主打商务风格，三折叠的设计更能满足商务人士在不同场景下的使用需求；10月，华为Mate全能旗舰系列将发布，该系列将拥有全能配置，涵盖芯片、影像 ...

[2] 华为发布会- 华为官网
华为发布会 ; 2026年四月 · 四月. 华为Pura 系列及全场景新品发布会 ; 2026年三月 · 三月. 华为春季全场景新品发布会 ; 2025年十二月 · 十二月. nova 15 系列及全场景新品发布会.

[3] 华为2026年手机发布规划曝光：华为三折叠或9月发布 - 腾讯新闻
9月，华为三折叠手机将发布。其主打商务风格，三折叠的设计更能满足商务人士在不同场景下的使用需求；10月，华为Mate全能旗舰系列将发布，该系列将拥有全能配置 ...
--- 第 2 步 ---
 正在调用 zai-org/GLM-4.5-Air 模型...
 大语言模型响应成功:
Thought: 根据搜索结果，我可以看到华为在2026年确实有手机发布计划。从搜索结果中，我可以看到以下信息：
1. 2026年9月：华为三折叠手机，主打商务风格，三折叠设计满足商务人士不同场景需求
2. 2026年10月：华为Mate全能旗舰系列，拥有全能配置，涵盖芯片、影像等
3. 2026年4月：华为Pura系列及全场景新品发布会


'华为在2026年发布了以下手机及其主要卖点：\n\n1. 华为Pura 70系列（2026年4月发布）\n   - 主要卖点：影像旗舰，搭载最新影像技术，包括更大传感器和更先进算法，提供专业级拍摄体验\n\n2. 华为三折叠手机（2026年9月发布）\n   - 主要卖点：商务风格，三折叠设计满足商务人士不同场景需求\n\n3. 华为Mate全能旗舰系列（2026年10月发布）\n   - 主要卖点：全能配置，涵盖芯片、影像等全方位顶级配置\n\n4. 华为nova系列新机（2026年3月春季发布会）\n   - 主要卖点：主打年轻市场，注重自拍功能和时尚设计\n\n这些产品覆盖了从影像旗舰到折叠屏，从商务全能到年轻时尚的各个细分市场，展现了华为在2026年的全面产品布局。'

## 需要你来使用LLM 和 React Agent来测试如下问题，分析和讨论两者回答问题结果的区别
1. 今天伦敦的最高气温和降雨概率分别是多少？
2. 苹果公司目前的 CEO 是谁，最新一季财报的营收是多少？
3. Deepseek当前最新模型版本叫什么，发布时间是什么时候？
4. "烟花三月下扬州"这个古诗是谁写的？
5.  最近一次世界杯亚洲区预选赛中国队的比分是多少？

In [11]:
import re
llm = HelloAgentsLLM()
tool_executor = ToolExecutor()
search_desc = "一个网页搜索引擎。当你需要回答关于时事、事实以及在你的知识库中找不到的信息时，应使用此工具。"
tool_executor.registerTool("Search", search_desc, search)
agent = ReActAgent(llm_client=llm, tool_executor=tool_executor)
question_list = ["今天伦敦的最高气温和降雨概率分别是多少","苹果公司目前的 CEO 是谁，最新一季财报的营收是多少？","Deepseek当前最新模型版本叫什么，发布时间是什么时候？","'烟花三月下扬州'这个古诗是谁写的？","最近一次世界杯亚洲区预选赛中国队的比分是多少？"]
for question in question_list:
    agent.run(question)

工具 'Search' 已注册。
--- 第 1 步 ---
 正在调用 zai-org/GLM-4.5-Air 模型...
 大语言模型响应成功:
Thought: 我需要查询今天伦敦的天气信息，包括最高气温和降雨概率。由于这是实时信息，我需要使用搜索引擎来获取最新的天气预报数据。

Action: Search[今天伦敦天气 最高气温 降雨概率]
思考: 我需要查询今天伦敦的天气信息，包括最高气温和降雨概率。由于这是实时信息，我需要使用搜索引擎来获取最新的天气预报数据。
 行动: Search[今天伦敦天气 最高气温 降雨概率]
 正在执行 [SerpApi] 网页搜索: 今天伦敦天气 最高气温 降雨概率
 观察: [1] 【伦敦天气】伦敦天气预报,天气预报一周,天气预报15天查询
周边地区 北京时2026-06-04 11:30更新 ; 阿伯丁. /. 11/12°C ; 阿伯丁小镇. /. 9/15°C ; 艾拉岛. /. 10/12°C ; 阿尔比马尔. /. 10/15°C ; 爱丁堡. /. 10/15°C ...

[2] 伦敦
1 月份 温度(°C) 降水(亳米) 伦敦 ... 日平均最高气温(°C), 平均总降水 (亳米), 平均降水日数. 一月, 3.1, 8.1, 41.6, 11.4. 二月, 2.7, 8.6, 36.3, 8.5. 三月, 4.6, 11.6 ...

[3] London天气状况：气温| 30天预报
今日预报显示最高气温19°C，最低气温13°C，截至05:30 PM 02 六月2026，全天温差为6 度。 London 今天的当前天气参数是什么？ 以下是截至05:30 PM 02 六月2026 London 当前天气 ...
--- 第 2 步 ---
 正在调用 zai-org/GLM-4.5-Air 模型...
 大语言模型响应成功:
Thought: 我需要从搜索结果中找到今天伦敦的最高气温和降雨概率信息。从观察结果来看，[3]提到了"今日预报显示最高气温19°C"，但没有明确提到降雨概率。[1]和[2]提供的是其他地区或历史平均数据，不是今天的具体信息。我需要更精确地搜索今天的降雨概率信息。

Action: Search[今天伦敦天气 降雨概率 2026年6月2日]
思考: 我需要从搜索结果中找到今天伦敦的

# 讨论
1. 大语言模型超参数如温度等指标，是否会对action步骤产生影响？
   会，LLMs输出随机性会影响action步骤，随机性越高，步骤会越多
2. 如何避免agent陷入多轮对话而无法终止？
   设置最大action step，到达最大step，LLMs自动总结累计对话给出答案
3. 如果让agent从多次对话中总结经验，使其在下一次对话中节省tokens的使用？
   可以使用agent memeory，让LLMs在每次回答问题前加载之前对话的经验，在LLMs每次回答后总结当次回答的经验并加载到agent memeory中
4. 实验过程中，出现大语言模型响应错误的情况时，你是如何解决的？
   设置回退等待机制
5. 设计更多的工具，如代码检测、新闻总结等，想一想是否可以将上一节课的RAG系统以工具形式结合到react agent中，增强agent的检索效率和准确率？
   在本次实验检索阶段，可以让agent调用rag作为工具，利用rag提供的更加精确的答案作为LLM回答的上下文

# 参考文献
[1] Yao S, Zhao J, Yu D, et al. React: Synergizing reasoning and acting in language models[C]//International Conference on Learning Representations (ICLR). 2023.